# RAG workshop — локальна версія: асистент ріелтора (Ollama, end-to-end)

Повністю локальний двійник [rag_workshop_02_realtor_assistant.ipynb](rag_workshop_02_realtor_assistant.ipynb): той самий корпус (`data/`) і той самий ланцюг **скан → chunking → embeddings → Chroma → retrieval → router → chat-history → відповідь із джерелами**, але без жодного зовнішнього API — усі виклики йдуть у локальний **Ollama**.

**Що є в цьому ноутбуці:**
- 01–05 — індексація корпусу (`.md`, `.csv`, `.pdf`) у ChromaDB, embeddings через Ollama (`nomic-embed-text`);
- 02a — опційний другий LLM-analyzer для PDF-договорів (розширені метадані);
- 06 — retrieval;
- 07 — **SQLite** для збереження історії чату (замість `dict` у пам'яті);
- 08–09 — **router** (вибір джерел) + **chat-history** разом, через Ollama `/api/chat`;
- 10 — **Gradio** UI + порівняння з іншими локальними інтерфейсами.

Evaluation тут свідомо не робимо — фокус на router + history + local UI.


### Схема локального пайплайна

![Local pipeline day 3](Local_pipeline_Day3.jpeg)


## Що потрібно

- Запущений **Ollama** локально (`ollama serve` — зазвичай стартує разом з застосунком).
- Завантажені моделі:
  - embedding: `nomic-embed-text` (`ollama pull nomic-embed-text`);
  - chat: `qwen2.5:3b-instruct` (`ollama pull qwen2.5:3b-instruct`) — обслуговує і router, і фінальну відповідь.
  - альтернатива: `gemma3:12b` — повільніша, але якісніша модель; варто спробувати, якщо router плутається у виборі джерел.
- **Без** `OPENAI_API_KEY` — весь пайплайн офлайн.
- **SQLite** (`sqlite3`) — стандартна бібліотека Python, нічого додатково встановлювати не треба.
- `chromadb`, `pymupdf`, `pandas`, `requests`, `gradio`.


In [ ]:
# Встановлюємо бібліотеки (без openai — замість нього requests до Ollama)
!pip install chromadb pymupdf pandas requests gradio -q


In [ ]:
import csv
import json
import re
import sqlite3
from pathlib import Path

import chromadb
import fitz  # PyMuPDF
import requests

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = PROJECT_ROOT / "chroma_data" / "realtor_assistant_local"
COLLECTION_NAME = "local_realtor_corpus_v1"
SQLITE_PATH = PROJECT_ROOT / "chroma_data" / "chat_history_local.db"  # окремий файл поруч із chroma_data

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "qwen2.5:3b-instruct"
#СHAT_MODEL = "gemma3:12b"  # альтернатива: повільніша, але якісніша модель (кращий router/відповіді)

MAX_HISTORY_TURNS = 4  # скільки пар user+assistant лишати в історії чату

print("DATA_DIR:", DATA_DIR)
print("CHROMA_DIR:", CHROMA_DIR)
print("SQLITE_PATH:", SQLITE_PATH)


## 01 - Збір корпусу документів з `data/`

Та сама логіка, що в дні 2: скануємо `data/` рекурсивно, беремо `.md`, `.csv`, `.pdf`.


In [ ]:
SUPPORTED_SUFFIXES = {".md", ".csv", ".pdf"}


def iter_corpus_files(root: Path) -> list[Path]:
    return [p for p in sorted(root.rglob("*")) if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES]


DATA_FILES = iter_corpus_files(DATA_DIR)
print(f"Знайдено файлів: {len(DATA_FILES)}")
for p in DATA_FILES:
    print(" ", p.relative_to(PROJECT_ROOT))


## 02 - Chunking і embeddings через Ollama

Chunking — той самий підхід (символи + overlap). Для embeddings `nomic-embed-text` очікує **task-префікс**: `search_document: ...` для того, що індексуємо, і `search_query: ...` для запиту — без цього якість retrieval помітно гірша.


In [ ]:
def chunker(text: str, chunk_size: int = 800, overlap: int = 80) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        piece = text[start : start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks


def ollama_embed(texts: list[str], is_query: bool = False) -> list[list[float]]:
    prefix = "search_query: " if is_query else "search_document: "
    payload = [prefix + t for t in texts]
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": EMBED_MODEL, "input": payload})
    return r.json()["embeddings"]


## 02a - (опційно) LLM-analyzer для метаданих PDF-договорів

`listings.csv`/`clients.csv` вже мають структуровані ціну/район/бюджет — фільтрувати по них через LLM не треба. А от у PDF-договорах (`data/contracts/`) корисні дані (тип угоди, комісія, дата, прізвище клієнта) заховані у вільному тексті. Другий, окремий LLM-виклик витягує їх у **filterable** поля для метаданих Chroma.

### Learning Piece: коли це важливо, а коли ні

**Важливо, коли:**
- реальні запити містять точні критерії саме по договорах ("комісія > X%", "договори 2025 року");
- договорів у `data/contracts/` уже десятки+, а не 2-3 демо-файли;
- дані в документах однорідні (майже всі містять ці поля).

**Не важливо / можна вимкнути, коли:**
- корпус малий — простіше знайти вручну або через `where={"source": "..."}`;
- документи різнорідні — багато `null`, фільтр стає ненадійним;
- критичний бюджет/latency — це ще один LLM-виклик на кожен документ;
- поле критично важливе (сума, дедлайн) — LLM може помилитись, тут краще людська перевірка.

Тому — прапорець `ENABLE_EXTRA_METADATA`, який можна вимкнути в один рядок.


In [ ]:
ENABLE_EXTRA_METADATA = True  # False -> пропустити другий LLM-analyzer (швидше й дешевше)


def extract_contract_extra_metadata(text: str, filename: str) -> dict:
    prompt = f"""Проаналізуй текст договору і поверни ТІЛЬКИ JSON з полями:
- deal_type (str | null): "sale" або "rent"
- client_surname (str | null): прізвище клієнта, ЛАТИНИЦЕЮ (транслітеровано, наприклад "Petrenko"), як воно зазвичай пишеться в назвах файлів договорів

Документ: {filename}
Текст:
---
{text[:6000]}
---
"""
    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": CHAT_MODEL, "messages": [{"role": "user", "content": prompt}], "stream": False},
    )
    raw = r.json()["message"]["content"].strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    try:
        data = json.loads(raw)
    except Exception:
        data = {}

    return {
        "deal_type": data.get("deal_type") or "",
        "client_surname": (data.get("client_surname") or "").strip().upper(),
    }


## 03 - Пофайлова обробка (`.md` / `.csv` / `.pdf`)

Той самий підхід, що в дні 2: кожен формат → чанки + метадані (`source`), CSV — один рядок = один документ, PDF — текст через PyMuPDF (+ опційно розширені метадані з кроку 02a).


In [ ]:
def process_md(path: Path) -> tuple[list[str], list[str], list[dict]]:
    text = path.read_text(encoding="utf-8").strip()
    parts = chunker(text)
    rel = str(path.relative_to(PROJECT_ROOT))
    base = rel.replace("/", "__")
    ids = [f"{base}:part{i:04d}" for i in range(len(parts))]
    metas = [{"source": rel} for _ in parts]
    return ids, parts, metas


def process_csv(path: Path) -> tuple[list[str], list[str], list[dict]]:
    rel = str(path.relative_to(PROJECT_ROOT))
    base = rel.replace("/", "__")
    ids, docs, metas = [], [], []

    with path.open(encoding="utf-8", newline="") as f:
        for row_idx, row in enumerate(csv.DictReader(f)):
            lines = [f"# Рядок {row_idx + 1} — {path.name}"]
            for k, v in row.items():
                if v:
                    lines.append(f"- **{k}**: {v}")
            parts = chunker("\n".join(lines))
            for i, chunk in enumerate(parts):
                ids.append(f"{base}__row{row_idx:04d}:part{i:04d}")
                docs.append(chunk)
                metas.append({"source": rel})

    return ids, docs, metas


def process_pdf(path: Path) -> tuple[list[str], list[str], list[dict]]:
    with fitz.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
    text = re.sub(r"\n{3,}", "\n\n", text.strip())
    parts = chunker(text)
    rel = str(path.relative_to(PROJECT_ROOT))
    base = rel.replace("/", "__")

    meta = {"source": rel}
    if ENABLE_EXTRA_METADATA:
        meta.update(extract_contract_extra_metadata(text, path.name))

    ids = [f"{base}:part{i:04d}" for i in range(len(parts))]
    metas = [meta for _ in parts]
    return ids, parts, metas


def upsert_batches(coll, ids: list[str], documents: list[str], metadatas: list[dict], batch_size: int = 64) -> None:
    for start in range(0, len(ids), batch_size):
        sl = slice(start, start + batch_size)
        embeddings = ollama_embed(documents[sl])
        coll.upsert(ids=ids[sl], documents=documents[sl], embeddings=embeddings, metadatas=metadatas[sl])


## 04 - Chroma: persistent-колекція

Нова назва колекції/директорії, бо `nomic-embed-text` (768-dim) несумісний з вектором OpenAI з дня 2 (3072-dim).


In [ ]:
FORCE_REBUILD = True  # True -> повна перебудова індексу

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if FORCE_REBUILD:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)


## 05 - Індексація

Проганяємо всі файли з `DATA_FILES` через відповідну `process_*`-функцію і завантажуємо в Chroma.


In [ ]:
if FORCE_REBUILD or collection.count() == 0:
    for path in DATA_FILES:
        suf = path.suffix.lower()
        if suf == ".md":
            ids, docs, metas = process_md(path)
        elif suf == ".csv":
            ids, docs, metas = process_csv(path)
        elif suf == ".pdf":
            ids, docs, metas = process_pdf(path)
        else:
            continue

        if docs:
            upsert_batches(collection, ids, docs, metas)
            print(f"OK {path.relative_to(PROJECT_ROOT)} → {metas[0]} ->{len(docs)} чанків")

    print("Готово. Чанків у колекції:", collection.count())
else:
    print("Колекція вже є; індексацію пропущено. Чанків:", collection.count())


## 06 - Retrieval

Той самий dense retrieval, що в дні 2: cosine distance, опційний `where`-фільтр.


In [ ]:
def retrieve(query: str, k: int = 6, where: dict | None = None):
    q_emb = ollama_embed([query], is_query=True)[0]
    return collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        where=where,
        include=["documents", "distances", "metadatas"],
    )


res = retrieve("Які квартири в оренду в районі Старе Місто?", k=5)
for dist, doc, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0]):
    print(f"[{dist:.3f}] {meta.get('source')}")
    print(doc[:200].replace("\n", " "))
    print("---")


## 07 - SQLite для chat-history

### Learning Piece: SQLite vs ChromaDB

Це два різні типи БД, і кожна відповідає за своє:

| | **ChromaDB** | **SQLite** |
|---|---|---|
| Тип | Векторна БД (similarity search) | Реляційна БД (SQL, таблиці/рядки) |
| Що зберігає | Ембеддинги чанків + текст + метадані | `chat_id`, `role`, `content`, `created_at` |
| Як шукає | Наближений пошук найближчих сусідів (cosine distance) | Точні SQL-запити (`WHERE`, `ORDER BY`, `LIMIT`) |
| Навіщо тут | Retrieval — знайти релевантні фрагменти `data/` | Пам'ять діалогу — останні N реплік конкретного `chat_id` |

**Цікавий факт:** Chroma сама всередині використовує SQLite для метаданих/документів (звідси `chroma.sqlite3` у `chroma_data/...`), але векторний пошук (HNSW-індекс) — окремий шар поверх неї. SQLite сам по собі **не вміє** similarity search за ембеддингами — тому дві бази тут не дублюють одна одну, а ділять відповідальність: Chroma = "що шукати в документах", SQLite = "що ми вже обговорили в цьому чаті".

Таблиця `messages(id, chat_id, role, content, created_at)`, файл — `SQLITE_PATH` (поруч із `chroma_data/`, окремо від колекції).


In [ ]:
def init_db() -> None:
    conn = sqlite3.connect(SQLITE_PATH)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            chat_id TEXT,
            role TEXT,
            content TEXT,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
        """
    )
    conn.commit()
    conn.close()


def get_history(chat_id: str, max_turns: int = MAX_HISTORY_TURNS) -> list[dict]:
    conn = sqlite3.connect(SQLITE_PATH)
    rows = conn.execute(
        "SELECT role, content FROM messages WHERE chat_id = ? ORDER BY id DESC LIMIT ?",
        (str(chat_id), max_turns * 2),
    ).fetchall()
    conn.close()
    return [{"role": role, "content": content} for role, content in reversed(rows)]


def append_history(chat_id: str, question: str, answer: str) -> None:
    conn = sqlite3.connect(SQLITE_PATH)
    conn.execute("INSERT INTO messages (chat_id, role, content) VALUES (?, 'user', ?)", (str(chat_id), question))
    conn.execute("INSERT INTO messages (chat_id, role, content) VALUES (?, 'assistant', ?)", (str(chat_id), answer))
    conn.commit()
    conn.close()


def reset_history(chat_id: str) -> None:
    conn = sqlite3.connect(SQLITE_PATH)
    conn.execute("DELETE FROM messages WHERE chat_id = ?", (str(chat_id),))
    conn.commit()
    conn.close()


init_db()


## Learning: many-filters у Chroma (на наших метаданих)

### Чому не працює "багато полів напряму"
`where` у Chroma має містити **один top-level оператор**.
Тому таке значення (багато ключів одразу) часто падає з помилкою валідації:
```python
where = {
  'deal_type': 'sale',
  'client_surname': 'Petrenko'
}
```

### Правильний спосіб: групувати через `$and` або `$or`
```python
where = {
  '$and': [
    {'deal_type': 'sale'},
    {'client_surname': 'Petrenko'}
  ]
}
```

Альтернатива (достатньо будь-якої умови):
```python
where = {
  '$or': [
    {'deal_type': 'sale'},
    {'deal_type': 'rent'}
  ]
}
```

### Filterable поля
Для прикладу обираємо лише **два** поля  лише в чанках PDF-договорів, і лише якщо `ENABLE_EXTRA_METADATA = True`:
- `deal_type` ("sale" | "rent")
- `client_surname` (рядок, **латиницею у ВЕРХНЬОМУ регістрі**, наприклад `"PETRENKO"` — нормалізовано в коді (`.strip().upper()`) і при індексації, і в router, щоб регістр/варіанти запису не ламали точний збіг)


### Які ще опції фільтрації є у Chroma
1. Логічні оператори: `$and`, `$or`
2. Оператори порівняння: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`
3. Оператори наборів: `$in`, `$nin`
4. Фільтрація по тексту документа через `where_document` (наприклад `$contains`)


In [ ]:
# Демонстрація: комбінований фільтр по розширених метаданих із кроку 02a
# client_surname зберігається і порівнюється у верхньому регістрі (нормалізовано в коді) — див. урок нижче
where_demo = {
    "$and": [
        {"deal_type": "sale"},
        {"client_surname": "PETRENKO"},
    ]
}

res_demo = retrieve("умови угоди з клієнтом", k=10, where=where_demo)
for dist, doc, meta in zip(res_demo["distances"][0], res_demo["documents"][0], res_demo["metadatas"][0]):
    print(f"[{dist:.3f}] {meta.get('source')} | deal_type={meta.get('deal_type')} | client_surname={meta.get('client_surname')}")


## 08 - Router: вибір джерел і метаданих (`where`) перед retrieval

Той самий принцип, що в дні 2 (Кейс 2), але тепер router обирає не лише `source`, а й (якщо доступні — крок 02a) `deal_type` та `client_surname` для PDF-договорів — точний збіг (`$eq`). Свідомо лише два прості поля, без числових/дата-порогів: менше полів — менше шансів на розбіжність значень між індексацією та запитом (див. урок у кроці 06.1). Це і є той момент, де LLM-analyzer із кроку 02a реально впливає на retrieval, а не просто зберігається невикористаним у метаданих.


In [ ]:
ALLOWED_SOURCES = sorted({str(p.relative_to(PROJECT_ROOT)) for p in DATA_FILES})

ROUTER_PROMPT = """Проаналізуй питання користувача і поверни ТІЛЬКИ JSON з полями:
- sources (list[str]): дозволені шляхи зі списку нижче, релевантні питанню; [] якщо звужувати не треба
- deal_type (str | null): "sale" або "rent" — ТІЛЬКИ якщо в питанні ЯВНО згадано продаж/купівлю чи оренду; за замовчуванням завжди null, навіть якщо йдеться про конкретного клієнта чи договір
- client_surname (str | null): прізвище клієнта з питання, ЛАТИНИЦЕЮ (транслітеровано, наприклад "Petrenko"), або null

Дозволені шляхи source:
{sources_list}

Підказки:
- каталог квартир / оголошення (пошук квартир, без конкретного клієнта чи договору) → data/listings.csv; deal_type тут НЕ заповнюй — це поле лише для PDF-договорів
- загальні відомості про клієнтів, записи про людей, угоди, договори → data/clients.csv
- конкретний договір, угода, contracts  (і інші синоніми) + прізвище клієнта → PDF з data/contracts/; заповнюй deal_type лише якщо в самому питанні є слово "продаж"/"купівля"/"sale" або "оренда"/"rent" — якщо просто просять "інформацію про клієнта та його угоди" без такого уточнення, deal_type = null (не вигадуй його)
- загальні правила (податки, політики, чеклисти .md) без явної вказівки "лише цей файл" → sources: []

Питання користувача: {question}

Поверни лише JSON, без пояснень.
"""


def _ollama_chat(messages: list[dict], temperature: float = 0.2) -> str:
    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": CHAT_MODEL, "messages": messages, "stream": False, "options": {"temperature": temperature}},
    )
    return r.json()["message"]["content"].strip()


def choose_filters_for_query(question: str) -> dict:
    prompt = ROUTER_PROMPT.format(sources_list="\n".join(f"- {s}" for s in ALLOWED_SOURCES), question=question)
    raw = _ollama_chat([{"role": "user", "content": prompt}], temperature=0)
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    try:
        data = json.loads(raw)
    except Exception:
        data = {}

    return {
        "sources": [s for s in data.get("sources") or [] if s in ALLOWED_SOURCES],
        "deal_type": data.get("deal_type") or None,
        "client_surname": ((data.get("client_surname") or "").strip().upper()) or None,
    }


def where_from_filters(filters: dict) -> dict | None:
    conditions = []

    sources = filters.get("sources") or []
    if len(sources) == 1:
        conditions.append({"source": sources[0]})
    elif len(sources) > 1:
        conditions.append({"source": {"$in": sources}})

    # deal_type/client_surname є лише в метаданих PDF-договорів (крок 02a).
    # Якщо джерела звужені й серед них немає жодного PDF з data/contracts/,
    # ці умови ніколи не збіжаться — тому застосовуємо їх лише коли це має сенс.
    sources_ok_for_contract_filters = not sources or any("contracts/" in s for s in sources)

    if ENABLE_EXTRA_METADATA and sources_ok_for_contract_filters:
        if filters.get("deal_type"):
            conditions.append({"deal_type": filters["deal_type"]})
        if filters.get("client_surname"):
            conditions.append({"client_surname": filters["client_surname"]})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}


### Покроковий приклад: питання -> фільтри -> формування WHERE

In [ ]:
question = "Які квартири в оренду в районі Старе Місто?"
#question = "Відомості про клієнта Petrenko та його угоди"
print("Question:", question)
filters = choose_filters_for_query(question=question)
print("Filters:", filters)
where_to_Chroma = where_from_filters(filters)
print("Where:", where_to_Chroma)

## 09 - Router + chat-history разом

Один фінальний кейс (без проміжних спрощень з дня 2): router обирає `where`, історія з SQLite додається як попередні `messages`, все йде в Ollama `/api/chat`.


In [ ]:
PROMPT_TEMPLATE = """Ти асистент ріелтора. Відповідай українською, спираючись лише на контекст нижче.
Якщо даних недостатньо — скажи прямо.
У кінці додай список джерел у форматі [source=...], лише ті, що реально використав.

Контекст:
{context}

Питання: {question}
Відповідь:"""


def format_hits(res) -> str:
    blocks = [f"[source={m.get('source')}]\n{d}" for d, m in zip(res["documents"][0], res["metadatas"][0])]
    return "\n\n---\n\n".join(blocks)


def rag_answer_with_router_history(question: str, k: int = 5, chat_id: str = "demo") -> str:
    history = get_history(chat_id)
    where = where_from_filters(choose_filters_for_query(question))
    res = retrieve(question, k=k, where=where)
    context = format_hits(res)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    messages = history + [{"role": "user", "content": prompt}]
    answer = _ollama_chat(messages, temperature=0.2)

    append_history(chat_id, question, answer)
    return answer


In [ ]:
# Демо-діалог: спочатку договори Петренко, потім уточнення лише по продажу (перевірка, що історія працює)
reset_history("demo")

print(rag_answer_with_router_history("Які умови по оплаті в договорах з Петренко?", chat_id="demo"))
print("\n---\n")
print(rag_answer_with_router_history("тільки по продажам", chat_id="demo"))

# Інші питання для проби (розкоментуйте потрібне):
# print(rag_answer_with_router_history("Покажи умови договору оренди з Петренко", chat_id="demo"))
# print(rag_answer_with_router_history("Які квартири в оренду в районі Старе Місто?", chat_id="demo"))
# print(rag_answer_with_router_history("Який відсоток податку на купівлю житла у 2026 році?", chat_id="demo"))
# print(rag_answer_with_router_history("Що ми домовились з Іваном Коваленком щодо бюджету?", chat_id="demo"))
# print(rag_answer_with_router_history("Які документи потрібні для зустрічі з кредитним радником?", chat_id="demo"))
# print(rag_answer_with_router_history("Покажи закриті угоди по всіх клієнтах", chat_id="demo"))


### Швидка перевірка кожного кроку за необхідності (мануальна, щоб зрозуміти що пішло не так)

In [ ]:
def rag_answer_with_router_history(question: str, k: int = 5, chat_id: str = "demo") -> str:
    history = get_history(chat_id)
    where = where_from_filters(choose_filters_for_query(question))
    print(("DEBUG: where filter:", where))
    res = retrieve(question, k=k, where=where)
    print(("DEBUG: retrieved hits:", res))
    context = format_hits(res)
    print(("DEBUG: formatted context:", context))
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    print(("DEBUG: prompt:", prompt))

    messages = history + [{"role": "user", "content": prompt}]
    answer = _ollama_chat(messages, temperature=0.2)
    print(("DEBUG: answer:", answer))

    append_history(chat_id, question, answer)
    return answer

In [ ]:
reset_history("demo")
print(rag_answer_with_router_history("Які умови по оплаті в договорах з Петренко?", chat_id="demo"))


In [ ]:
print(rag_answer_with_router_history("Які квартири в оренду в районі Старе Місто?", chat_id="demo"))

## 10 - User Interface: локально, через Gradio

**Чи точно Telegram працює локально?** І так, і ні:
- Сам **бот-процес** може працювати повністю на вашій машині, без публічного IP чи сервера — через **polling**, а не webhook. У цьому сенсі "локально" — так, все обчислення (Ollama, Chroma, SQLite) на вашому залізі.
- Але **сам Telegram — хмарний сервіс**: бот все одно ходить в інтернет до серверів Telegram API, щоб отримати повідомлення й надіслати відповідь. Якщо мета — **справді офлайн** (без інтернету взагалі) — Telegram **не підходить**.
- Тобто Telegram ОК, якщо мета — "дані й обчислення лишаються локальними" (приватність), але **не** ОК, якщо мета — "працює без інтернету".

**Альтернативи:**

| Інструмент | Складність | Результат | Коли обирати |
|---|---|---|---|
| **Gradio** | ⭐ мінімальна | Веб-чат у браузері на `localhost` | Найшвидше демо, повністю офлайн |
| **Streamlit** | ⭐⭐ невелика | Веб-додаток з UI на `localhost` | Більше контролю над дизайном/станом сесії |
| **FastAPI** | ⭐⭐⭐ середня | REST API на `localhost` | Інтеграція з іншим фронтендом/системою |
| **Open WebUI** | ⭐ мінімальна | Готовий ChatGPT-подібний UI поверх Ollama | Вже є Ollama, потрібен UI без коду |
| **Telegram Bot (polling)** | ⭐⭐ невелика | Бот у Telegram, зручно з телефону | Демо для команди/клієнта; **потребує інтернет** до Telegram API |

**Обрали Gradio** для цього ноутбука — найменше коду, повністю в браузері, без залежності від інтернету. Повний приклад Telegram-бота (polling) — дивись `rag_workshop_03.ipynb`, розділ 08.


In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=lambda message, history: rag_answer_with_router_history(message, k=10, chat_id="gradio_session"),
    title="Асистент ріелтора (локально, Ollama)",
)

demo.launch() 


---

## 🎓 Learning Piece — "Локальний" RAG: що це насправді означає?

> Одне з найважливіших питань перед деплоєм: **чи дійсно мої документи ізольовані?**

### "Локальний" — це не одне поняття

Слово **"локальний"** у контексті RAG може означати дві різні речі, і їх легко переплутати:

**1. Локальний = документи не йдуть до OpenAI / Anthropic / Google**
Це головна мотивація. Ollama запускає модель на твоєму залізі — жоден документ не відправляється через API третьої сторони. Саме це більшість людей мають на увазі, кажучи "локальний RAG".

**2. Локальний = фізично на моєму комп'ютері**
Це вужче визначення. Якщо деплоїш на VPS — це вже чужий фізичний сервер, хоча ОС і дані під твоїм контролем.

---

### Рівні ізольованості (від найбільшого до найменшого)

```
✅  Твій ноутбук / сервер у офісі
      → ніхто не має фізичного доступу, повна ізоляція

✅  Приватний сервер у корпоративній мережі
      → твоя інфраструктура, твій контроль

⚠️  VPS у хмарного провайдера (Hetzner, DigitalOcean, AWS)
      → ти контролюєш ОС і дані, але не фізичне залізо
      → провайдер теоретично має доступ до диску

❌  OpenAI API + Pinecone / Weaviate Cloud
      → документи обробляються на серверах третіх сторін
      → потрапляють у умови використання чужого сервісу
```

---

### Як обирати рівень ізольованості?

| Use case | Рекомендація |
|---|---|
| Внутрішня база знань компанії (не секретна) | ✅ VPS достатньо |
| Асистент ріелтора з даними клієнтів | ✅ VPS + шифрування диску |
| Медичні дані, юридичні договори (GDPR / HIPAA) | ⚠️ Приватний сервер або on-premise |
| Держсектор, оборонна промисловість | ❌ Тільки air-gapped, без інтернету |

---

### Що захищає локальний RAG, а що — ні

| Загроза | Локальний RAG захищає? |
|---|---|
| Витік документів через API OpenAI | ✅ Так — модель не бачить твої дані |
| Треті сторони тренують модель на твоїх даних | ✅ Так — Ollama не відправляє нічого |
| Злом VPS-сервера хакером | ⚠️ Ні — потрібен захист на рівні сервера |
| Юридичний запит до провайдера VPS | ⚠️ Залежить від юрисдикції провайдера |
| Фізичний доступ до залізa | ✅ Так (якщо сервер у тебе) / ⚠️ Ні (якщо VPS) |

---

### TL;DR

> **Локальний RAG** = модель та дані не йдуть до OpenAI та аналогів.  
> Це вже **великий крок** для приватності порівняно з хмарними LLM.  
>  
> Але якщо твої документи **юридично чутливі** — VPS не достатньо.  
> Тоді потрібен **власний фізичний сервер** або **on-premise інфраструктура**.  
>  
> Питай себе: *"Хто ще, крім мене, теоретично може побачити ці дані?"*  
> Якщо відповідь неприйнятна — піднімай рівень ізольованості.